# Image Feature Extraction **[DEMO]**

Determine which device to run PyTorch on:
- CUDA if an Nvidia GPU is installed
- CPU otherwise

In [22]:
import torch
from torch.nn.functional import cosine_similarity

# Set device to CUDA if we have an Nvidia GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {device}")

Using device cuda


---
## Example 1

Load image datasets

In [23]:
from PIL import Image
import requests

img_urls = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.jpeg",
]
image_real = Image.open(requests.get(img_urls[0], stream=True).raw).convert("RGB")
image_gen = Image.open(requests.get(img_urls[1], stream=True).raw).convert("RGB")

In [24]:
cow1 = Image.open(r"demo_imgs/Cow1.jpg")
cow2 = Image.open(r"demo_imgs/Cow2.jpg")

Import pretrained image processor and ML model

In [25]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 8251.25it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Basic inference function

In [26]:
def infer(image):
    inputs = processor(image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    return outputs.pooler_output

Pass images to inference function to obtain embeddings

In [27]:
embed_real = infer(image_real)
embed_gen = infer(image_gen)

In [28]:
embed_cow1 = infer(cow1)
embed_cow2 = infer(cow2)

Calculate similarity scores

In [29]:
similarity_score = cosine_similarity(embed_real, embed_gen, dim=1)

similarity_score = cosine_similarity(embed_real, embed_real, dim=1)
print(similarity_score)

tensor([1.], device='cuda:0', grad_fn=<SumBackward1>)


In [30]:
cow_sim_score = cosine_similarity(embed_cow1, embed_cow2, dim=1)
print(cow_sim_score)

tensor([0.6996], device='cuda:0', grad_fn=<SumBackward1>)
